In [1]:
import pandas as pd

print("1. Читаем огромную таблицу платежей...")
installments = pd.read_csv('../data/raw/installments_payments.csv')
print(f"Размер таблицы: {installments.shape[0]} миллионов строк!")

print("\n2. Создаем новую фичу: Дни просрочки (DPD - Days Past Due)...")
# DAYS_ENTRY_PAYMENT - когда реально заплатил
# DAYS_INSTALMENT - когда должен был заплатить
# Если разница > 0, значит просрочил!
installments['DPD'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']
installments['DPD'] = installments['DPD'].apply(lambda x: x if x > 0 else 0)

print("3. Группируем данные по каждому клиенту...")
# Считаем для каждого клиента:
# - Максимальную просрочку в днях
# - Среднюю просрочку
# - Общее количество платежей
pay_agg = installments.groupby('SK_ID_CURR', as_index=False).agg({
    'DPD':['max', 'mean'],
    'NUM_INSTALMENT_VERSION': 'count'
})

# Схлопываем двухэтажные колонки
pay_agg.columns =['SK_ID_CURR', 'MAX_PAST_DUE_DAYS', 'MEAN_PAST_DUE_DAYS', 'TOTAL_PAYMENTS']

print("\nУра! Новые мощные фичи готовы:")
display(pay_agg.head())

1. Читаем огромную таблицу платежей...
Размер таблицы: 13605401 миллионов строк!

2. Создаем новую фичу: Дни просрочки (DPD - Days Past Due)...
3. Группируем данные по каждому клиенту...

Ура! Новые мощные фичи готовы:


,SK_ID_CURR,MAX_PAST_DUE_DAYS,MEAN_PAST_DUE_DAYS,TOTAL_PAYMENTS
0,100001,11.0,1.571429,7
1,100002,0.0,0.000000,19
2,100003,0.0,0.000000,25
3,100004,0.0,0.000000,3
4,100005,1.0,0.111111,9


In [2]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

print("4. Читаем главную анкету клиентов...")
train = pd.read_csv('../data/raw/application_train.csv')

print("5. Приклеиваем фичи просрочек к главной таблице...")
# Объединяем по колонке SK_ID_CURR (ID клиента)
train_merged = train.merge(pay_agg, on='SK_ID_CURR', how='left')
print(f"Размер новой таблицы: {train_merged.shape}")

print("\n6. Готовим данные к обучению...")
X = train_merged.drop(columns=['SK_ID_CURR', 'TARGET'])
y = train_merged['TARGET']

# Обрабатываем категории для LightGBM
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].astype('category')

# ВАЖНО: У новых клиентов может не быть истории платежей, у них появятся пустые значения (NaN).
# Заполним пропуски медианой, как делали раньше.
num_cols = X.select_dtypes(exclude=['category']).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

print("7. Разделяем данные и обучаем новую модель...")
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, n_jobs=-1)

model.fit(
    X_train, y_train, 
    eval_set=[(X_val, y_val)], 
    eval_metric='auc', 
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

print("\n8. Оцениваем результат...")
y_pred_proba = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred_proba)
print(f"\n🚀 СТАРЫЙ ROC AUC БЫЛ: 0.7599")
print(f"🌟 НОВЫЙ ROC AUC СТАЛ: {auc:.4f}")

4. Читаем главную анкету клиентов...
5. Приклеиваем фичи просрочек к главной таблице...
Размер новой таблицы: (307511, 125)

6. Готовим данные к обучению...
7. Разделяем данные и обучаем новую модель...
[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022192 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12048
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 119
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[290]	valid_0's auc: 0.763064	valid_0's binary_logloss: 0.244341

8. Оцениваем результат...

🚀 СТАРЫЙ ROC AUC БЫЛ: 0.7599
🌟 НОВЫЙ ROC AUC С

In [3]:
import joblib

print("1. Сохраняем новую, поумневшую модель...")
joblib.dump(model, '../models/lgbm_baseline.pkl')

print("2. Обновляем схему данных (теперь в ней 125 колонок!)...")
# Берем 1 строчку из нашей НОВОЙ таблицы с просрочками и перезаписываем schema.csv
train_merged.head(1).to_csv('../models/schema.csv', index=False)

print("✅ Готово! Мозг и Схема обновлены.")

1. Сохраняем новую, поумневшую модель...
2. Обновляем схему данных (теперь в ней 125 колонок!)...
✅ Готово! Мозг и Схема обновлены.
